In [1]:
import pandas as pd 
import numpy as np 
import parquet

import geopandas as gpd
import requests

import seaborn as sns 
import matplotlib.pyplot as plt 

import re
import string
import ftfy

# STATIONS

In [2]:
raw_stations = pd.read_json("raw_stations.json")

In [3]:
raw_stations[raw_stations.start_station_id==7008.0]

,start_station_id,start_station_name
50,7008.0,Wellesley St / Queen's Park Cres
8491029,7008.0,Wellesley St / Queen's Park Cres
22676041,7008.0,Wellesley St W / Queen's Park Cres


In [4]:
raw_stations.dropna(how='all').shape

(2675, 2)

In [5]:
raw_stations.columns = ['station_id', 'name']

In [6]:
raw_stations.isna().sum()

station_id    271
name          243
dtype: int64

In [7]:
(raw_stations
    .dropna(how='all')
    .groupby('station_id')
    .size()
    .reset_index(name='number')
    .sort_values('number', ascending=False)
    .query('number>1')
    )

,station_id,number
70,7070.0,9
330,7358.0,8
454,7492.0,5
120,7123.0,5
163,7171.0,5
...,...,...
115,7118.0,2
456,7494.0,2
467,7510.0,2
455,7493.0,2


In [8]:
# each of 864 sation ids has 2 or more name versions

## Clean IDs in stations vocab.

### fix_encoding - data collection problem, bad encoding symbols in the raw data

In [8]:
# GÇô
KNOWN_MOJIBAKE = {'GÇô': '–', 'GÇÖ': "'"}

def fix_encoding(s):
    if pd.isna(s):
        return s
    s = str(s)
    for _ in range(5):
        fixed = ftfy.fix_text(s)
        if fixed == s:
            break
        s = fixed
    for bad, good in KNOWN_MOJIBAKE.items():
        s = s.replace(bad, good)
    return s


In [9]:
raw_stations['name'] = [fix_encoding(name) for name in raw_stations['name']]

In [10]:
raw_stations.shape

(2675, 2)

In [11]:
raw_stations = raw_stations.drop_duplicates()
raw_stations.shape

(2065, 2)

In [12]:
raw_stations[raw_stations.station_id.isna()]

,station_id,name
333353,NaN,Princess St / Adelaide St E
333354,NaN,Fort York Blvd / Capreol Crt
333356,NaN,Elizabeth St / Edward St (Bus Terminal)
333358,NaN,Bay St / St. Joseph St
333359,NaN,Bay St / College St (East Side)
...,...,...
666280,NaN,Vanauley St / Queen St W - SMART
668360,NaN,Summerhill Ave / MacLennan Ave - SMART
668447,NaN,Bloor St W / Shaw Ave - SMART
802678,NaN,Exhibition GO Station


### assign the most rescent version of name

In [13]:
raw_stations[raw_stations.station_id==7008.0]['name']

50            Wellesley St / Queen's Park Cres
22676041    Wellesley St W / Queen's Park Cres
Name: name, dtype: str

In [14]:
raw_stations[raw_stations.station_id==7008.0].groupby('station_id')['name'].transform(
    lambda s: s.dropna().iloc[-1] if s.notna().any() else pd.NA
)

50          Wellesley St W / Queen's Park Cres
22676041    Wellesley St W / Queen's Park Cres
Name: name, dtype: str

In [15]:
has_id = raw_stations['station_id'].notna()
raw_stations['name'] = raw_stations.loc[has_id].groupby('station_id')['name'].transform(
    lambda s: s.dropna().iloc[-1] if s.notna().any() else pd.NA
)

In [16]:
raw_stations = raw_stations.drop_duplicates()
raw_stations[raw_stations.station_id==7008.0]['name']

50    Wellesley St W / Queen's Park Cres
Name: name, dtype: object

In [30]:
# raw_stations[raw_stations.name.isna()]

In [17]:
raw_stations = raw_stations.dropna(how='all')

In [18]:
raw_stations.isna().sum()

station_id    0
name          5
dtype: int64

In [19]:
(raw_stations
    .dropna(how='all')
    .groupby('station_id')
    .size()
    .reset_index(name='number')
    .sort_values('number', ascending=False)
    .query('number>1')
    )

,station_id,number


In [36]:
# raw_stations[raw_stations.station_id.isna()][:4]

## Normalize NAMEs in stations vocab

### strip_trailing_id_suffix

In [20]:
def strip_trailing_id_suffix(s):
    if pd.isna(s):
        return s
    return re.sub(r"\s*-\s*\d{4,5}$", "", str(s)).strip()

In [21]:
raw_stations['name'] = [strip_trailing_id_suffix(name) for name in raw_stations.name]

### standardize_intersection

In [22]:
def standardize_intersection(s, sep='/'):
    if pd.isna(s):
        return s
    s = str(s)
    if sep not in s:
        return s
    parts = [p.strip() for p in s.split(sep)]
    parts.sort(key=str.lower)
    return f' {sep} '.join(parts)

In [23]:
raw_stations['normalized_name'] = [standardize_intersection(name) for name in raw_stations.name]

In [24]:
raw_stations = raw_stations.drop_duplicates()

In [25]:
raw_stations.isna().sum()

station_id         0
name               5
normalized_name    5
dtype: int64

### delete punctuation

In [26]:
EXTRA_PUNCT = "\u2019\u2013\u2014\u00b4\u201c\u201d"  # ’ – — ´ " "

def delete_punctuation(name):
    if pd.isna(name):
        return name
    s = fix_encoding(name)
    s = re.sub(rf"[{re.escape(string.punctuation)}{EXTRA_PUNCT}\x00-\x1f\x7f]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s 


In [27]:
raw_stations['normalized_name'] = [delete_punctuation(name) for name in raw_stations['normalized_name']]

In [28]:
(raw_stations
    .dropna(how='all')
    .groupby('station_id')
    .size()
    .reset_index(name='number')
    .sort_values('number', ascending=False)
    .query('number>1')
    )

,station_id,number


In [238]:
raw_stations[raw_stations.name=='Joe Shuster Way King St W']

,station_id,name,normalized_name


In [ ]:
# mask = ~(
#     raw_stations['station_id'].isna() & 
#     raw_stations.groupby('name')['station_id'].transform(lambda x: x.notna().any())
# )

# raw_stations = raw_stations[mask]

In [48]:
raw_stations[raw_stations.name=='Joe Shuster Way King St W']

,station_id,name,normalized_name


In [29]:
raw_stations.isna().sum()

station_id         0
name               5
normalized_name    5
dtype: int64

### add is_smart variable and remove from the name

In [30]:
raw_stations.head()

,station_id,name,normalized_name
0,7073.0,Spadina Ave / Fort York Blvd,Fort York Blvd Spadina Ave
1,7038.0,Dundas St W / Yonge St,Dundas St W Yonge St
2,7012.0,Elizabeth St / Edward St (Bus Terminal),Edward St Bus Terminal Elizabeth St
3,7172.0,Strachan Ave / Princes' Blvd,Princes Blvd Strachan Ave
4,7039.0,Simcoe St / Michael Sweet Ave - SMART,Michael Sweet Ave SMART Simcoe St


In [31]:
def remove_prefix_smart(name):
    if pd.isna(name):
        return name
    
    if name.lower().__contains__('smart'):
        name = re.sub(r'-?\s*smart', ' ', name, flags=re.IGNORECASE)
        name = re.sub(r'\s+', ' ', name).strip(' -')
        
    return name
        
raw_stations['is_smart'] = raw_stations.name.str.lower().str.contains('smart').astype('int16')

In [32]:
s = re.sub('SMART', " ", 'Dundas St W SMART Howard Park Ave')
re.sub(r"\s+", ' ', s)

'Dundas St W Howard Park Ave'

In [243]:
str('Spadina Ave Sussex Ave SMART'.lower()).__contains__('smart')

True

In [244]:
remove_prefix_smart('SMARTSMARTsmart Spadina smart Ave SMART Sussex Ave SMART')

'Spadina Ave Sussex Ave'

In [33]:
raw_stations['normalized_name'] = [remove_prefix_smart(name) for name in raw_stations.normalized_name]

In [ ]:
# raw_stations[raw_stations.name.isna()]

In [34]:

duplicate_names = raw_stations.groupby('name', as_index=False).size().query('size > 1')['name']

raw_stations[raw_stations.name.isin(duplicate_names)].sort_values('name')

,station_id,name,normalized_name,is_smart
152,7051.0,Mutual St / Shuter St,Mutual St Shuter St,0
12531117,7693.0,Mutual St / Shuter St,Mutual St Shuter St,0
3415783,7398.0,York St / Lake Shore Blvd W,Lake Shore Blvd W York St,0
3490901,7470.0,York St / Lake Shore Blvd W,Lake Shore Blvd W York St,0


### normalize street/directions

In [35]:
STREET_TYPE_MAP = {
    'street': 'st', 'avenue': 'ave', 'boulevard': 'blvd', 'road': 'rd',
    'drive': 'dr', 'place': 'pl', 'crescent': 'cres', 'court': 'ct', 'crt': 'ct',
    'lane': 'ln', 'terrace': 'terr',
}
DIRECTION_MAP = {'west': 'w', 'east': 'e', 'north': 'n', 'south': 's'}

# def _replace_words(s, mapping):
#     if pd.isna(s):
#         return s
#     return re.sub(r"[A-Za-z]+", lambda m: mapping.get(m.group(0).lower(), m.group(0)), str(s))

def _replace_words(s, mapping):
    if pd.isna(s):
        return s
    def repl(m):
        word = m.group(0)
        replacement = mapping.get(word.lower())
        if replacement is None:
            return word
        return replacement.capitalize() if word[0].isupper() else replacement
    return re.sub(r"[A-Za-z]+", repl, str(s))

def normalize_street_types(s):
    return _replace_words(s, STREET_TYPE_MAP)

def normalize_directions(s):
    return _replace_words(s, DIRECTION_MAP)

In [36]:
raw_stations['normalized_name'] = [normalize_street_types(name) for name in raw_stations.normalized_name]
raw_stations['normalized_name'] = [normalize_directions(name) for name in raw_stations.normalized_name]

In [37]:
raw_stations = raw_stations.drop_duplicates()

In [38]:
raw_stations.isna().sum()

station_id         0
name               5
normalized_name    5
is_smart           0
dtype: int64

### fuzzy matching

In [39]:
raw_stations[raw_stations.station_id.isna()].sort_values('name')

,station_id,name,normalized_name,is_smart


In [103]:
from difflib import get_close_matches, SequenceMatcher

def tokenize(s):
    return re.findall(r"[A-Za-z0-9']+", s.lower())

def is_safe_match(a, b):
    set1, set2 = set(tokenize(a)), set(tokenize(b))
    unmatched1, unmatched2 = list(set1 - set2), list(set2 - set1)
    if not unmatched1 or not unmatched2:
        return True
    return all(
        max((SequenceMatcher(None, w, w2).ratio() for w2 in unmatched2), default=0) >= 0.6
        for w in unmatched1
    )

known_names = raw_stations.loc[raw_stations['station_id'].notna(), 'name'].dropna().unique().tolist()
missing_mask = raw_stations['station_id'].isna()
missing_names = raw_stations.loc[missing_mask, 'name'].unique()

fuzzy_resolved = {}
for name in missing_names:
    candidates = get_close_matches(name, known_names, n=5, cutoff=0.82)
    safe_candidates = [c for c in candidates if is_safe_match(name, c)]
    matched_ids = raw_stations.loc[raw_stations['name'].isin(safe_candidates), 'station_id'].dropna().unique()
    if len(matched_ids) == 1:
        fuzzy_resolved[name] = matched_ids[0]

raw_stations.loc[missing_mask, 'station_id'] = (
    raw_stations.loc[missing_mask, 'name'].map(fuzzy_resolved)
    .fillna(raw_stations.loc[missing_mask, 'station_id'])
)


In [242]:
dupl = raw_stations.groupby('station_id', as_index=False).size().query('size>1')['station_id']

# GEODATA

In [64]:
# fetch station_information from Toronto Bike Share's public API (active right now)
url = "https://tor.publicbikesystem.net/ube/gbfs/v1/en/station_information"
stations = pd.json_normalize(requests.get(url).json()["data"]["stations"])
stations = stations[["station_id", "name", "lat", "lon"]].copy()
stations['station_id'] = pd.to_numeric(stations['station_id'], errors='coerce')

## Live GBFS feed 

In [ ]:
# # transform stations to gdf

# stations_gdf = gpd.GeoDataFrame(
#     stations,
#     geometry=gpd.points_from_xy(stations['lon'], stations['lat']),
#     crs='EPSG:4326'
# )


In [5]:
# requests.get(url).json()

## Historical snapshots

In [ ]:
# url = "https://web.archive.org/cdx/search/cdx?url=tor.publicbikesystem.net/ube/gbfs/v1/en/station_information&output=json"

# Historical snapshots via the Wayback Machine — we queried web.archive.org's CDX API to find every time that same 
# live endpoint had been archived, found 15 captures spanning 2017–2026, and fetched each one's JSON the same way as t
# he live feed. Unioned all 15 + the live feed, deduplicated by station_id (keeping the most recent name/coordinates per ID) — 
# this is the stations/all_stations table everything else builds on.

In [ ]:
# query web.archive.org's CDX API to find every to find archived station_information (found 15 captures spanning 2017–2026)
timestamps = [
    '20170726114519','20190819094031','20191021144817','20210507022246',
    '20211027025026','20220307133912','20220706122732','20230326051048',
    '20240413054106','20241004201519','20250425175804','20250817130624',
    '20251010082450','20260305164528','20260310030434'
]

import time

def fetch_snapshot(ts, retries=2, delay=1.0):
    url = f"https://web.archive.org/web/{ts}if_/https://tor.publicbikesystem.net/ube/gbfs/v1/en/station_information"
    resp = None
    for attempt in range(retries + 1):
        resp = requests.get(url, timeout=30)
        if resp.status_code == 200 and resp.text.strip():
            try:
                return resp.json()["data"]["stations"]
            except (ValueError, KeyError):
                pass
        time.sleep(delay * (attempt + 1))
    raise RuntimeError(f"failed after {retries + 1} attempts (last status {resp.status_code if resp is not None else 'n/a'})")

hist_frames = []
for ts in timestamps:
    try:
        data = fetch_snapshot(ts)
        frame = pd.json_normalize(data)[["station_id", "name", "lat", "lon"]]
        hist_frames.append(frame)
        print(ts, "->", len(frame), "stations")
    except Exception as e:
        print(ts, "-> FAILED:", e)
    time.sleep(1)  # be polite between snapshots regardless of outcome


In [8]:
parsed_stations = pd.concat(hist_frames + [stations], ignore_index=True)
parsed_stations["station_id"] = pd.to_numeric(parsed_stations["station_id"], errors="coerce")
# all_stations = all_stations.dropna(subset=["station_id", "lat", "lon"]).drop_duplicates(subset="station_id", keep="last")
# all_stations.shape
parsed_stations = parsed_stations.dropna(how='all')



In [ ]:
# parsed_stations.to_json("parsed_stations.json")

### Connect raw_stations and parsed_stations with geodata

In [40]:
parsed_stations = pd.read_json('parsed_stations.json')
parsed_stations.head()

,station_id,name,lat,lon
0,7000,Ft. York / Capreol Crt.,43.639832,-79.395954
1,7001,Lower Jarvis St / The Esplanade,43.647992,-79.370907
2,7002,St George St / Bloor St W,43.667333,-79.399429
3,7003,Madison Ave / Bloor St W,43.667158,-79.402761
4,7004,University Ave / Elm St,43.656518,-79.389099


In [41]:
(raw_stations
    .merge(parsed_stations, how='outer', on='station_id', indicator=True)
    .query('_merge=="left_only"')['station_id'].unique()
    )

array([7218., 7219., 7393., 7394., 7511., 7532., 7573., 7690., 7714.,
       7722., 7793., 7795., 7905., 8019., 8024., 8067., 8071., 8072.,
       8097., 8098., 8100., 8103., 8116., 8118., 8121., 8124., 8126.,
       8127., 8132., 8135., 8136., 8146., 8222., 8233.])

In [100]:
raw_stations.columns

Index(['station_id', 'name', 'normalized_name', 'is_smart'], dtype='str')

In [101]:
parsed_stations.columns

Index(['station_id', 'name', 'lat', 'lon'], dtype='str')

In [107]:
print(raw_stations['station_id'].duplicated().any())
print(parsed_stations['station_id'].duplicated().any())

False
True


In [95]:
# pd.concat([raw_stations, parsed_stations.drop(columns='name')]).drop_duplicates(subset='station_id', keep='first')

In [ ]:
(raw_stations
    .merge(parsed_stations.rename(columns={'name': 'name_parsed'}), how='left')
    .query('name.isna() & name_parsed.notna()')
    )

,station_id,name,normalized_name,is_smart,name_parsed,lat,lon
5241,7683.0,NaN,NaN,0,Bay St / Harbour St (West),43.642101,-79.378453
5242,7683.0,NaN,NaN,0,Bay St / Harbour St (West),43.641910,-79.378325
5243,7683.0,NaN,NaN,0,Bay St / Harbour St (West),43.641910,-79.378325
5244,7683.0,NaN,NaN,0,Bay St / Harbour St (West),43.641910,-79.378325
5345,7697.0,NaN,NaN,0,St Clair Ave W / Cloverdale Rd - SMART Vault,43.673925,-79.461543


In [170]:
# raw_stations = raw_stations.drop('normalized_name', axis=1)

In [42]:
all_stations = (raw_stations
    .sort_values('station_id')
    .merge(parsed_stations.rename(columns={'name': 'name_parsed'}), how='outer', on='station_id')
    .drop_duplicates(subset=['station_id'])
    .assign(name = lambda d: d['name'].combine_first(d['name_parsed']))
    # .query('name.isna() & name_parsed.notna()')
    .drop('name_parsed', axis=1)
    # .drop('normalized_name', axis=1)
    )

In [125]:
# all_stations.groupby('station_id', as_index=False).size().query('size>1')

In [43]:
all_stations[[str(name).lower().__contains__('russell') for name in all_stations.name]]

,station_id,name,normalized_name,is_smart,lat,lon
3868,7465.0,Russell Hill Rd / St. Clair Ave W,Russell Hill Rd St Clair Ave W,0.0,43.685569,-79.408019


In [175]:
all_stations.isna().sum()

station_id     0
name           3
is_smart       0
lat           34
lon           34
dtype: int64

In [44]:
all_stations = all_stations.reset_index(drop=True)

In [63]:
def remove_prefix_smart(name):
    if pd.isna(name):
        return name
    
    if name.lower().__contains__('smart'):
        name = re.sub(r'-?\s*smart', ' ', name, flags=re.IGNORECASE)
        name = re.sub(r'\s+', ' ', name).strip(' -')
        
    return name
        
all_stations['is_smart'] = all_stations.name.str.lower().str.contains('smart').astype('int16')

In [64]:
all_stations['name'] = [remove_prefix_smart(name) for name in all_stations.name]

In [65]:
all_stations.isna().sum()

station_id          0
name                3
normalized_name     7
is_smart            0
lat                10
lon                10
dtype: int64

In [66]:
unmatched_geocodes = all_stations[all_stations.name.notna() & all_stations.lat.isna()]['name'].values.tolist()
unmatched_geocodes[:5]

['Make Invisible',
 'Wolfpack',
 'Toronto Bike Shop',
 'PBSC-OPS',
 "135 Queen's Wharf"]

In [201]:
all_stations[all_stations['lat'].isna()][:4]

,station_id,name,is_smart,lat,lon
201,7218.0,Make Invisible,0,NaN,NaN
202,7219.0,Wolfpack,0,NaN,NaN
363,7393.0,55 Magnificent Rd. Garage,0,NaN,NaN
364,7394.0,Toronto Bike Shop,0,NaN,NaN


In [67]:
from utils import geocode_names

geocoded = geocode_names(unmatched_geocodes)  # ~1 req/sec, so this takes a while

missing_mask = all_stations['lat'].isna() & all_stations['name'].notna()
coords = all_stations.loc[missing_mask, 'name'].map(geocoded)


In [68]:
geocoded

{"135 Queen's Wharf": (43.6495303, -79.3698418),
 'Charles St / Church St': (43.6685473, -79.3859605)}

In [70]:
all_stations.loc[missing_mask, 'lat'] = coords.map(lambda c: c[0] if isinstance(c, tuple) else np.nan)
all_stations.loc[missing_mask, 'lon'] = coords.map(lambda c: c[1] if isinstance(c, tuple) else np.nan)

In [52]:
all_stations[:4]

,station_id,name,normalized_name,is_smart,lat,lon
0,7000.0,Fort York Blvd / Capreol Ct,Capreol Ct Fort York Blvd,0.0,43.639832,-79.395954
1,7001.0,Wellesley Station Green P,Wellesley Station Green P,0.0,43.647992,-79.370907
2,7002.0,St. George St / Bloor St W,Bloor St W St George St,0.0,43.667333,-79.399429
3,7003.0,Madison Ave / Bloor St W,Bloor St W Madison Ave,0.0,43.667158,-79.402761


In [53]:
unmatched_geocodes = all_stations[all_stations.name.notna() & all_stations.lat.isna()]['name'].values.tolist()
unmatched_geocodes

['Make Invisible',
 'Wolfpack',
 '55 Magnificent Rd. Garage',
 'Toronto Bike Shop',
 'PBSC-OPS',
 "135 Queen's Wharf - SMART",
 'Torrens Ave / Broadview Ave',
 'Charles St / Church St - SMART',
 'Palmerston Blvd / Herrick St',
 'Scrivener Sq / Yonge St',
 'Concord Ave / Bloor St W (South)',
 'Duncanwoods Dr / Finch Ave W',
 'Pearldale Ave / Finch Ave W',
 'Tobermory Dr / Finch Ave W',
 'Finch Ave W / Weston Rd',
 'Albion Rd / Finch Ave W',
 'Martin Grove Rd / Finch Ave W',
 'Westmore Dr / Finch Ave W',
 'Humber College Blvd / Arboretum Blvd',
 'Eglinton Ave E / Swift Dr',
 'Islington Ave / Aviemore Dr',
 'Rumike Rd / Finch Ave W',
 'George St S / The Esplanade',
 'NycLab',
 'St. Andrew Subway Station']

In [54]:
manual_geocodes = {
    'Torrens Ave / Broadview Ave': (43.690257, -79.3551454),
    'Palmerston Blvd / Herrick St': (43.6621834, -79.4126711),
    'Scrivener Sq / Yonge St': (43.6805292, -79.3909585),
    'Concord Ave / Bloor St W (South)': (43.66189, -79.42679),  
    'Duncanwoods Dr / Finch Ave W': (43.7489179, -79.5576785),
    'Pearldale Ave / Finch Ave W': (43.7479089, -79.562385),
    'Tobermory Dr / Finch Ave W': (43.7592118, -79.5078974),
    'Finch Ave W / Weston Rd': (43.7519497, -79.5424358),
    'Albion Rd / Finch Ave W': (43.7411965, -79.589314),
    'Martin Grove Rd / Finch Ave W': (43.7367192, -79.5915358),
    'Westmore Dr / Finch Ave W': (43.7345848, -79.6009496),
    'Humber College Blvd / Arboretum Blvd': (43.7299356, -79.6036041),
    'Eglinton Ave E / Swift Dr': (43.7262179, -79.3183042),
    'Islington Ave / Aviemore Dr': (43.7483345, -79.5683495),
    'Rumike Rd / Finch Ave W': (43.749881, -79.5522663),
    'George St S / The Esplanade': (43.6484943, -79.369733),
    'St. Andrew Subway Station': (43.6477917, -79.3848711),
    '55 Magnificent Rd. Garage': (43.61282, -79.51171),
}

In [60]:
missing_mask = all_stations['lat'].isna() & all_stations['name'].notna()
coords = all_stations.loc[missing_mask, 'name'].map(manual_geocodes)
all_stations.loc[missing_mask, 'lat'] = coords.map(lambda c: c[0] if isinstance(c, tuple) else np.nan)
all_stations.loc[missing_mask, 'lon'] = coords.map(lambda c: c[1] if isinstance(c, tuple) else np.nan)

In [71]:
all_stations.isna().sum()

station_id         0
name               3
normalized_name    7
is_smart           0
lat                8
lon                8
dtype: int64

In [72]:
unmatched_geocodes = all_stations[all_stations.name.notna() & all_stations.lat.isna()]['name'].values.tolist()
unmatched_geocodes

['Make Invisible', 'Wolfpack', 'Toronto Bike Shop', 'PBSC-OPS', 'NycLab']

In [ ]:
# all_stations = all_stations.drop('normalized_name', axis=1)

In [75]:
stations_gdf = gpd.GeoDataFrame(
    all_stations,
    geometry=gpd.points_from_xy(all_stations['lon'], all_stations['lat']),
    crs='EPSG:4326'
)



## NEIGHBOURHOODS

In [76]:
neigh = gpd.read_file('toronto-neighbourhoods.geojson')[['AREA_NAME', 'geometry']]

neigh_pts = neigh.copy()
neigh_pts['geometry'] = neigh_pts.representative_point()

neigh.head()


,AREA_NAME,geometry
0,South Eglinton-Davisville,"MULTIPOLYGON (((-79.38635 43.69783, -79.38623 ..."
1,North Toronto,"MULTIPOLYGON (((-79.39744 43.70693, -79.39837 ..."
2,Dovercourt Village,"MULTIPOLYGON (((-79.43411 43.66015, -79.43537 ..."
3,Junction-Wallace Emerson,"MULTIPOLYGON (((-79.4387 43.66766, -79.43841 4..."
4,Yonge-Bay Corridor,"MULTIPOLYGON (((-79.38404 43.64497, -79.38502 ..."


In [77]:
wards = gpd.read_file('toronto-wards.geojson')[['AREA_NAME', 'geometry']].rename(columns={'AREA_NAME': 'ward'})
wards.head()

,ward,geometry
0,Humber River-Black Creek,"MULTIPOLYGON (((-79.49105 43.7635, -79.49008 4..."
1,York Centre,"MULTIPOLYGON (((-79.44043 43.7634, -79.43998 4..."
2,Willowdale,"MULTIPOLYGON (((-79.39449 43.76157, -79.39461 ..."
3,University-Rosedale,"MULTIPOLYGON (((-79.39004 43.6905, -79.39004 4..."
4,Beaches-East York,"MULTIPOLYGON (((-79.29864 43.71515, -79.29837 ..."


In [78]:
stations_gdf = gpd.sjoin(stations_gdf, neigh[['AREA_NAME', 'geometry']], how='left', predicate='within')
stations_gdf = stations_gdf.drop(columns='index_right').rename(columns={'AREA_NAME': 'neighbourhood'})

stations_gdf = gpd.sjoin(stations_gdf, wards[['ward', 'geometry']], how='left', predicate='within')
stations_gdf = stations_gdf.drop(columns='index_right')
stations_gdf

,station_id,name,is_smart,lat,lon,geometry,neighbourhood,ward
0,7000.0,Fort York Blvd / Capreol Ct,0,43.639832,-79.395954,POINT (-79.39595 43.63983),Harbourfront-CityPlace,Spadina-Fort York
1,7001.0,Wellesley Station Green P,0,43.647992,-79.370907,POINT (-79.37091 43.64799),St Lawrence-East Bayfront-The Islands,Spadina-Fort York
2,7002.0,St. George St / Bloor St W,0,43.667333,-79.399429,POINT (-79.39943 43.66733),University,University-Rosedale
3,7003.0,Madison Ave / Bloor St W,0,43.667158,-79.402761,POINT (-79.40276 43.66716),Annex,University-Rosedale
4,7004.0,University Ave / Elm St,0,43.656518,-79.389099,POINT (-79.3891 43.65652),Kensington-Chinatown,University-Rosedale
...,...,...,...,...,...,...,...,...
1197,8269.0,Greystone Walk Dr / Midland Ave,0,43.724207,-79.252430,POINT (-79.25243 43.72421),Kennedy Park,Scarborough Southwest
1198,8271.0,R.C. Harris Treatment Plant,0,43.674894,-79.278307,POINT (-79.27831 43.67489),Birchcliffe-Cliffside,Scarborough Southwest
1199,8272.0,Kipling Station,0,43.636674,-79.537380,POINT (-79.53738 43.63667),Etobicoke City Centre,Etobicoke-Lakeshore
1200,8273.0,Exhibition Place _ Coca Cola Coliseum,0,43.634427,-79.415647,POINT (-79.41565 43.63443),Fort York-Liberty Village,Spadina-Fort York


In [81]:
stations_gdf = stations_gdf.drop(['geometry'], axis=1)

In [82]:
stations_gdf.isna().sum()

station_id       0
name             3
is_smart         0
lat              8
lon              8
neighbourhood    8
ward             8
dtype: int64

In [83]:
stations_gdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 1202 entries, 0 to 1201
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   station_id     1202 non-null   float64
 1   name           1199 non-null   str    
 2   is_smart       1202 non-null   int16  
 3   lat            1194 non-null   float64
 4   lon            1194 non-null   float64
 5   neighbourhood  1194 non-null   str    
 6   ward           1194 non-null   str    
dtypes: float64(3), int16(1), str(3)
memory usage: 128.7 KB


# PARQUET STORAGE

In [84]:
from pathlib import Path

# Hive-style partitioning by city: data/<table>/city=<city>/part.parquet.
# Adding a new city later is just writing another `city=<new_city>` folder -
# no schema change, and pd.read_parquet(DATA_DIR / "stations") reads all cities at once.
DATA_DIR = Path("data")
CITY = "toronto"

def city_path(table):
    path = DATA_DIR / table / f"city={CITY}"
    path.mkdir(parents=True, exist_ok=True)
    return path / "part.parquet"

In [ ]:
stations_out = stations_gdf.astype({
    'station_id': 'Int32',
    'is_smart': 'int8',
    'lat': 'float32',
    'lon': 'float32',
    'neighbourhood': 'category',
    'ward': 'category',
})
stations_out.to_parquet(city_path("stations"), index=False, compression='zstd')


neigh_out = neigh.rename(columns={'AREA_NAME': 'neighbourhood'})
neigh_out.to_parquet(city_path("neighbourhoods"), index=False, compression='zstd')

wards_out = wards.copy()
wards_out.to_parquet(city_path("wards"), index=False, compression='zstd')

In [ ]:
stations_check = pd.read_parquet(DATA_DIR / "stations")
print(stations_check.shape, stations_check['city'].unique())
print(stations_check.dtypes[['neighbourhood', 'ward']])

neigh_check = gpd.read_parquet(city_path("neighbourhoods"))
wards_check = gpd.read_parquet(city_path("wards"))
neigh_check.head()

(1202, 8) ['toronto']
Categories (1, str): ['toronto']
neighbourhood    category
ward             category
dtype: object


,neighbourhood,geometry
0,South Eglinton-Davisville,"MULTIPOLYGON (((-79.38635 43.69783, -79.38623 ..."
1,North Toronto,"MULTIPOLYGON (((-79.39744 43.70693, -79.39837 ..."
2,Dovercourt Village,"MULTIPOLYGON (((-79.43411 43.66015, -79.43537 ..."
3,Junction-Wallace Emerson,"MULTIPOLYGON (((-79.4387 43.66766, -79.43841 4..."
4,Yonge-Bay Corridor,"MULTIPOLYGON (((-79.38404 43.64497, -79.38502 ..."
